# 03. Batch Predictions With Prefect

This notebook shows how the registered MLflow model is reused in a batch workflow. Instead of retraining, we load the model from MLflow and score a whole parquet file with the Prefect flow used by the project.

Before you run it, finish [02-train-ml-model.ipynb](02-train-ml-model.ipynb) so MLflow already has at least one registered model version to resolve.


## Workflow

```mermaid
flowchart LR
    A["Batch input parquet"] --> B["Prefect flow"]
    B --> C["Resolve latest model from MLflow"]
    C --> D["Score every ride record"]
    D --> E["Write prediction parquet"]
```


In [ ]:
from pathlib import Path

import pandas as pd

from src.batch.flow import score_batch_flow
from src.common.config import get_settings

settings = get_settings()
output_path = Path("data/predictions/notebook_batch_predictions.parquet")
settings

## Run One Local Batch Flow

The flow reads the input parquet file, resolves the latest registered model in MLflow, and writes a prediction parquet file to the local `data/predictions/` directory.

In [ ]:
result_path = score_batch_flow(
    input_uri=settings.batch_input_uri,
    output_path=str(output_path),
)
result_path

In [ ]:
predictions = pd.read_parquet(result_path)
predictions.head()

## Interpret The Batch Output

Because this sample input still contains the observed trip duration, the output file includes both actual and predicted values. That gives us a quick way to check whether the batch run looks reasonable.

In [ ]:
summary = pd.DataFrame(
    {
        "rows": [len(predictions)],
        "mean_actual_duration": [predictions["actual_duration"].mean()],
        "mean_predicted_duration": [predictions["predicted_duration"].mean()],
        "mean_absolute_error": [
            (predictions["actual_duration"] - predictions["predicted_duration"])
            .abs()
            .mean()
        ],
    }
)
summary

## Add One Helpful Comparison

Averages can hide a few large misses. The next cell surfaces the biggest absolute errors so you can inspect whether those rows look unusual.

In [ ]:
largest_errors = predictions.assign(
    absolute_error=(
        predictions["actual_duration"] - predictions["predicted_duration"]
    ).abs()
).sort_values("absolute_error", ascending=False)

largest_errors[
    [
        "ride_id",
        "PULocationID",
        "DOLocationID",
        "trip_distance",
        "actual_duration",
        "predicted_duration",
        "absolute_error",
    ]
].head(10)

## Visual Check: Actual vs Predicted Duration

A quick scatter plot gives a better intuition than a summary statistic alone. Points close to the diagonal line indicate trips where the batch prediction was close to the observed duration.

In [ ]:
import matplotlib.pyplot as plt

sample_for_plot = predictions.sample(min(len(predictions), 800), random_state=42)
ax = sample_for_plot.plot.scatter(
    x="actual_duration",
    y="predicted_duration",
    alpha=0.25,
    figsize=(6, 6),
    title="Batch predictions: actual vs predicted duration",
)
limits = [
    min(
        sample_for_plot["actual_duration"].min(),
        sample_for_plot["predicted_duration"].min(),
    ),
    max(
        sample_for_plot["actual_duration"].max(),
        sample_for_plot["predicted_duration"].max(),
    ),
]
ax.plot(limits, limits, linestyle="--", color="black", linewidth=1)
ax.set_xlim(limits)
ax.set_ylim(limits)
plt.show()

## Schedule The Same Flow

The notebook runs the flow once. To keep it on a schedule, use the project helper below in a terminal:

```bash
python -m src.batch.serve
```

That command starts a Prefect deployment using the cron expression from `.env`. It keeps running so Prefect can poll for scheduled runs; press `Ctrl+C` when you want to stop serving the deployment. If it fails, confirm that the local Prefect server is running and that `PREFECT_API_URL` still matches your local stack.
